
# Buy Fee Model V8.2 — Low-Memory Matched Control + DiD

## Why this version exists

Earlier V8 versions crashed because the weekly panel was too large to:
- load fully into RAM,
- or repeatedly group inside Python for many event dates.

V8.2 uses a safer two-stage design:

1. **Stage A — reduce the weekly panel to only rows that fall inside any relevant event window.**
   - Reads the weekly CSV in small chunks.
   - Keeps only required columns.
   - Writes filtered rows to disk immediately.
   - Does not keep the full weekly panel in memory.

2. **Stage B — aggregate the filtered rows by event date and segment.**
   - Uses one event-window lookup table.
   - Works from the reduced file.
   - Produces treated/control snapshots.

3. **Stage C — matched controls + Difference-in-Differences.**

This is slower than an in-memory workflow but much safer on a normal laptop.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import statsmodels.formula.api as smf

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V61_DIR = DATA_FOLDER / "buy_fee_model_dataset_v6_1"

WEEKLY_FILE = V61_DIR / "04_WEEKLY_MODEL_PANEL_V6_1.csv"
EVENTS_FILE = V61_DIR / "05_CLEAN_FEE_CHANGE_EVENTS_V6_1.csv"
STRICT_FILE = V61_DIR / "09_MODEL_READY_STRICT_V6_1.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILTERED_WEEKLY_FILE = OUTPUT_DIR / "00_FILTERED_WEEKLY_EVENT_WINDOWS.csv"

PRIMARY_WINDOW = 8
MIN_OBSERVED_WEEKS = 6
MIN_REGISTERED_PRE = 25
MIN_REGISTERED_POST = 25

N_CONTROLS = 3
MAX_MATCH_DISTANCE = 4.0
PRICE_TOLERANCE = 3000
EXCLUDE_SAME_LOCATION = True
CONTROL_CHANGE_EXCLUSION_WEEKS = 8

CHUNK_SIZE = 25000

print("Weekly file exists:", WEEKLY_FILE.exists())
print("Events file exists:", EVENTS_FILE.exists())
print("Strict file exists:", STRICT_FILE.exists())

if not (WEEKLY_FILE.exists() and EVENTS_FILE.exists() and STRICT_FILE.exists()):
    raise FileNotFoundError("One or more required V6.1 files are missing.")


Weekly file exists: True
Events file exists: True
Strict file exists: True


## 1. Load small event files

In [2]:

events = pd.read_csv(
    EVENTS_FILE,
    parse_dates=["BUYFEE_CHANGE_DATE"],
    low_memory=True
)

strict = pd.read_csv(
    STRICT_FILE,
    parse_dates=["BUYFEE_CHANGE_DATE", "Event_Week"],
    low_memory=True
)

for df in [events, strict]:
    df["VRLOCATION"] = df["VRLOCATION"].astype("string").fillna("").str.strip().str.upper()
    df["LANE"] = df["LANE"].astype("string").fillna("").str.strip().str.upper()
    df["price_tier"] = df["price_tier"].astype("string").fillna("").str.strip()

print("Strict events:", len(strict))
print("Strict locations:", strict["VRLOCATION"].nunique())
print("Strict corporate dates:", strict["Corporate_Change_ID"].nunique())


Strict events: 1843
Strict locations: 35
Strict corporate dates: 37


## 2. Build event-window calendar table

In [3]:

change_dates = (
    strict[["Corporate_Change_ID", "BUYFEE_CHANGE_DATE"]]
    .drop_duplicates()
    .sort_values("BUYFEE_CHANGE_DATE")
    .reset_index(drop=True)
)

window_rows = []

for _, r in change_dates.iterrows():
    dt = pd.Timestamp(r["BUYFEE_CHANGE_DATE"])
    event_week = dt - pd.to_timedelta(dt.weekday(), unit="D")

    for rel in range(-PRIMARY_WINDOW, PRIMARY_WINDOW + 1):
        if rel == 0:
            continue

        week = event_week + pd.Timedelta(weeks=rel)

        window_rows.append({
            "Corporate_Change_ID": r["Corporate_Change_ID"],
            "BUYFEE_CHANGE_DATE": dt,
            "Event_Week": event_week,
            "Week_Start": week,
            "Period": "PRE" if rel < 0 else "POST",
            "Relative_Week": rel,
        })

window_calendar = pd.DataFrame(window_rows)

window_calendar.to_csv(
    OUTPUT_DIR / "01_EVENT_WINDOW_CALENDAR.csv",
    index=False
)

needed_weeks = set(window_calendar["Week_Start"].dt.strftime("%Y-%m-%d"))

print("Corporate dates:", change_dates["Corporate_Change_ID"].nunique())
print("Unique weekly dates needed:", len(needed_weeks))


Corporate dates: 37
Unique weekly dates needed: 171



## 3. Stage A — create filtered weekly file on disk

This is the key memory fix.

The cell reads only the columns needed by V8 and only retains weeks that are part of at least one event window.


In [5]:
# ============================================================
# STAGE A — CREATE FILTERED WEEKLY FILE
# MEMORY SAFE + PANDAS/ARROW SAFE
# ============================================================

WEEKLY_USECOLS = [
    "Week_Start",
    "VRLOCATION",
    "LANE",
    "price_tier",
    "Registered_Units",
    "Sold_Units",
    "BuyFee_Revenue",
    "Sold_With_BuyFee",
    "SaleAmt_Final_Sum",
    "SaleAmt_Final_Count",
    "Vehicle_Age_Sum",
    "Vehicle_Age_Count",
    "Mileage_Sum",
    "Mileage_Count",
    "Condition_Sum",
    "Condition_Count",
    "High_Bid_Sum",
    "High_Bid_Count",
    "Floor_Sum",
    "Floor_Count",
    "Conversion",
]


# ============================================================
# SETTINGS
# ============================================================

# If your computer has limited RAM, 10,000–25,000 is safer.
# This does NOT change the analytical results.
SAFE_CHUNK_SIZE = 20_000


# ============================================================
# REMOVE OLD PARTIAL OUTPUT
# ============================================================

if FILTERED_WEEKLY_FILE.exists():

    FILTERED_WEEKLY_FILE.unlink()

    print(
        "Removed previous partial filtered file:"
    )

    print(
        FILTERED_WEEKLY_FILE
    )


# ============================================================
# INITIALIZE COUNTERS
# ============================================================

header_written = False

rows_read = 0

rows_written = 0

chunks_processed = 0

chunks_with_matches = 0


# ============================================================
# VALIDATE NEEDED WEEKS
# ============================================================

if len(needed_weeks) == 0:

    raise ValueError(
        "needed_weeks is empty. "
        "Run the event-window calendar cell first."
    )


print("=" * 80)

print(
    "CREATING MEMORY-SAFE FILTERED WEEKLY FILE"
)

print("=" * 80)

print(
    "Source file:"
)

print(
    WEEKLY_FILE
)

print(
    "\nOutput file:"
)

print(
    FILTERED_WEEKLY_FILE
)

print(
    "\nWeekly dates needed:",
    f"{len(needed_weeks):,}"
)

print(
    "Chunk size:",
    f"{SAFE_CHUNK_SIZE:,}"
)


# ============================================================
# CREATE CHUNK READER
# ============================================================

reader = pd.read_csv(

    WEEKLY_FILE,

    usecols=WEEKLY_USECOLS,

    chunksize=SAFE_CHUNK_SIZE,

    low_memory=True,

    encoding="utf-8",

    encoding_errors="replace"
)


# ============================================================
# PROCESS WEEKLY FILE
# ============================================================

for chunk_num, chunk in enumerate(
    reader,
    start=1
):

    chunks_processed += 1

    rows_read += len(chunk)


    # ========================================================
    # 1. PARSE WEEK START SAFELY
    # ========================================================

    ws = pd.to_datetime(
        chunk["Week_Start"],
        errors="coerce"
    )


    # --------------------------------------------------------
    # Convert parsed dates to standard YYYY-MM-DD strings.
    #
    # IMPORTANT:
    # We intentionally keep Week_Start as STRING in this
    # temporary CSV.
    #
    # The next notebook cell will parse it back to datetime.
    # --------------------------------------------------------

    week_str = (
        ws
        .dt.strftime("%Y-%m-%d")
    )


    # ========================================================
    # 2. KEEP ONLY EVENT-WINDOW WEEKS
    # ========================================================

    mask = (
        week_str
        .isin(needed_weeks)
        .fillna(False)
    )


    if not mask.any():

        # Free objects immediately

        del (
            chunk,
            ws,
            week_str,
            mask
        )

        gc.collect()


        if chunk_num % 20 == 0:

            print(
                f"Chunks={chunk_num:,} | "
                f"Rows read={rows_read:,} | "
                f"Rows written={rows_written:,}"
            )


        continue


    # ========================================================
    # 3. CREATE SMALL FILTERED DATAFRAME
    # ========================================================

    # Resetting index here makes all later assignments safe.
    #
    # We avoid:
    #
    # small.loc[:, "Week_Start"] = datetime_array
    #
    # because pandas Arrow string columns can reject datetime
    # assignment.

    small = (
        chunk
        .loc[
            mask,
            WEEKLY_USECOLS
        ]
        .reset_index(
            drop=True
        )
    )


    # ========================================================
    # 4. REPLACE WEEK_START SAFELY
    # ========================================================

    # Keep as normalized string in temporary CSV.
    # Direct column replacement allows dtype replacement safely.

    kept_week_str = (
        week_str
        .loc[mask]
        .reset_index(
            drop=True
        )
    )

    small["Week_Start"] = (
        kept_week_str
        .astype("string")
    )


    # ========================================================
    # 5. NORMALIZE LOCATION
    # ========================================================

    small["VRLOCATION"] = (

        small["VRLOCATION"]

        .astype("string")

        .fillna("")

        .str.strip()

        .str.upper()
    )


    # ========================================================
    # 6. NORMALIZE LANE
    # ========================================================

    small["LANE"] = (

        small["LANE"]

        .astype("string")

        .fillna("")

        .str.strip()

        .str.upper()
    )


    # ========================================================
    # 7. NORMALIZE PRICE TIER
    # ========================================================

    small["price_tier"] = (

        small["price_tier"]

        .astype("string")

        .fillna("")

        .str.strip()
    )


    # ========================================================
    # 8. FORCE NUMERIC MODEL FIELDS TO NUMERIC
    # ========================================================

    numeric_cols = [

        "Registered_Units",

        "Sold_Units",

        "BuyFee_Revenue",

        "Sold_With_BuyFee",

        "SaleAmt_Final_Sum",

        "SaleAmt_Final_Count",

        "Vehicle_Age_Sum",

        "Vehicle_Age_Count",

        "Mileage_Sum",

        "Mileage_Count",

        "Condition_Sum",

        "Condition_Count",

        "High_Bid_Sum",

        "High_Bid_Count",

        "Floor_Sum",

        "Floor_Count",

        "Conversion",
    ]


    for col in numeric_cols:

        if col in small.columns:

            small[col] = pd.to_numeric(
                small[col],
                errors="coerce"
            )


    # ========================================================
    # 9. REMOVE INVALID KEY ROWS
    # ========================================================

    small = small[

        small["Week_Start"].notna()

        & small["VRLOCATION"].ne("")

        & small["LANE"].ne("")

        & small["price_tier"].ne("")

    ]


    if small.empty:

        del (
            chunk,
            ws,
            week_str,
            mask,
            kept_week_str,
            small
        )

        gc.collect()

        continue


    # ========================================================
    # 10. APPEND TO FILTERED CSV ON DISK
    # ========================================================

    small.to_csv(

        FILTERED_WEEKLY_FILE,

        mode="a",

        header=not header_written,

        index=False,

        encoding="utf-8"
    )


    header_written = True

    rows_written += len(
        small
    )

    chunks_with_matches += 1


    # ========================================================
    # 11. MEMORY CLEANUP
    # ========================================================

    del (
        chunk,
        ws,
        week_str,
        mask,
        kept_week_str,
        small
    )

    gc.collect()


    # ========================================================
    # 12. PROGRESS
    # ========================================================

    if chunk_num % 20 == 0:

        print(
            f"Chunks={chunk_num:,} | "
            f"Rows read={rows_read:,} | "
            f"Rows written={rows_written:,}"
        )


# ============================================================
# FINAL VALIDATION
# ============================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FILTERED WEEKLY FILE COMPLETE"
)

print(
    "=" * 80
)

print(
    "Chunks processed:",
    f"{chunks_processed:,}"
)

print(
    "Chunks containing required weeks:",
    f"{chunks_with_matches:,}"
)

print(
    "Rows read:",
    f"{rows_read:,}"
)

print(
    "Rows written:",
    f"{rows_written:,}"
)

print(
    "\nFiltered file:"
)

print(
    FILTERED_WEEKLY_FILE
)


# ============================================================
# VERIFY OUTPUT FILE
# ============================================================

if not FILTERED_WEEKLY_FILE.exists():

    raise FileNotFoundError(
        "Filtered weekly file was not created."
    )


if rows_written == 0:

    raise ValueError(
        "Filtered weekly file contains zero rows. "
        "Check Week_Start values and needed_weeks."
    )


file_size_mb = (
    FILTERED_WEEKLY_FILE.stat().st_size
    / (1024 ** 2)
)

print(
    "\nFiltered file size:",
    f"{file_size_mb:,.2f} MB"
)


# ============================================================
# SMALL SAMPLE VALIDATION
# ============================================================

sample_check = pd.read_csv(

    FILTERED_WEEKLY_FILE,

    nrows=10,

    low_memory=True
)


print(
    "\nOutput columns:"
)

print(
    sample_check.columns.tolist()
)


print(
    "\nSample:"
)

display(
    sample_check
)


print(
    "\n"
    + "=" * 80
)

print(
    "STAGE A PASSED"
)

print(
    "=" * 80
)

print(
    "Continue to the next notebook cell."
)

CREATING MEMORY-SAFE FILTERED WEEKLY FILE
Source file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_dataset_v6_1\04_WEEKLY_MODEL_PANEL_V6_1.csv

Output file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v8_2_outputs\00_FILTERED_WEEKLY_EVENT_WINDOWS.csv

Weekly dates needed: 171
Chunk size: 20,000
Chunks=20 | Rows read=400,000 | Rows written=345,542
Chunks=40 | Rows read=800,000 | Rows written=691,053
Chunks=60 | Rows read=1,200,000 | Rows written=1,034,090
Chunks=80 | Rows read=1,600,000 | Rows written=1,379,145

FILTERED WEEKLY FILE COMPLETE
Chunks processed: 85
Chunks containing required weeks: 85
Rows read: 1,689,856
Rows written: 1,456,729

Filtered file:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Fi

,Week_Start,VRLOCATION,LANE,price_tier,Registered_Units,Sold_Units,BuyFee_Revenue,Sold_With_BuyFee,SaleAmt_Final_Sum,SaleAmt_Final_Count,...,Vehicle_Age_Count,Mileage_Sum,Mileage_Count,Condition_Sum,Condition_Count,High_Bid_Sum,High_Bid_Count,Floor_Sum,Floor_Count,Conversion
0,2023-01-02,ALBANY,A,0-1000,2,2,0.0,0,1050.0,2,...,2,287597.0,2,0.0,0,1050,2,0,2,1.000000
1,2023-01-09,ALBANY,A,0-1000,1,1,0.0,0,500.0,1,...,1,173958.0,1,0.0,0,500,1,0,1,1.000000
2,2023-01-16,ALBANY,A,0-1000,2,0,0.0,0,1700.0,2,...,2,337592.0,2,0.0,0,1700,2,0,2,0.000000
3,2023-01-23,ALBANY,A,0-1000,1,1,0.0,0,500.0,1,...,1,217009.0,1,0.0,0,500,1,0,1,1.000000
4,2023-01-30,ALBANY,A,0-1000,1,1,0.0,0,500.0,1,...,1,181118.0,1,0.0,0,500,1,0,1,1.000000
5,2023-02-06,ALBANY,A,0-1000,1,0,0.0,0,700.0,1,...,1,160732.0,1,0.0,0,700,1,0,1,0.000000
6,2023-02-13,ALBANY,A,0-1000,2,1,0.0,0,1500.0,2,...,2,360094.0,2,0.0,0,1500,2,0,2,0.500000
7,2023-02-20,ALBANY,A,0-1000,1,1,0.0,0,700.0,1,...,1,297860.0,1,0.0,0,700,1,0,1,1.000000
8,2023-03-06,ALBANY,A,0-1000,3,2,0.0,0,2300.0,3,...,3,511382.0,3,0.0,0,2300,3,0,3,0.666667
9,2023-03-13,ALBANY,A,0-1000,2,1,0.0,0,1300.0,2,...,2,264954.0,2,0.0,0,1300,2,0,2,0.500000



STAGE A PASSED
Continue to the next notebook cell.


## 4. Stage B — load the reduced weekly file

In [6]:

filtered_weekly = pd.read_csv(
    FILTERED_WEEKLY_FILE,
    parse_dates=["Week_Start"],
    low_memory=True
)

print("Filtered weekly rows:", f"{len(filtered_weekly):,}")
print("Locations:", filtered_weekly["VRLOCATION"].nunique())
print("Date range:", filtered_weekly["Week_Start"].min(), "to", filtered_weekly["Week_Start"].max())


Filtered weekly rows: 1,456,729
Locations: 53
Date range: 2022-12-26 00:00:00 to 2026-03-30 00:00:00


## 5. Expand filtered rows to event windows

In [7]:

# Join on Week_Start so each weekly row is attached only to
# corporate event windows that include that calendar week.
expanded = filtered_weekly.merge(
    window_calendar,
    on="Week_Start",
    how="inner"
)

del filtered_weekly
gc.collect()

print("Expanded rows:", f"{len(expanded):,}")
print("Corporate dates:", expanded["Corporate_Change_ID"].nunique())


Expanded rows: 5,094,556
Corporate dates: 37


## 6. Aggregate PRE/POST snapshots

In [8]:

group_cols = [
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_DATE",
    "Event_Week",
    "Period",
    "VRLOCATION",
    "LANE",
    "price_tier"
]

agg = (
    expanded.groupby(
        group_cols,
        dropna=False,
        sort=False
    )
    .agg(
        observed_weeks=("Week_Start", "nunique"),
        registered=("Registered_Units", "sum"),
        sold=("Sold_Units", "sum"),

        buyfee_revenue=("BuyFee_Revenue", "sum"),
        sold_with_buyfee=("Sold_With_BuyFee", "sum"),

        saleamt_sum=("SaleAmt_Final_Sum", "sum"),
        saleamt_count=("SaleAmt_Final_Count", "sum"),

        age_sum=("Vehicle_Age_Sum", "sum"),
        age_count=("Vehicle_Age_Count", "sum"),

        mileage_sum=("Mileage_Sum", "sum"),
        mileage_count=("Mileage_Count", "sum"),

        condition_sum=("Condition_Sum", "sum"),
        condition_count=("Condition_Count", "sum"),

        highbid_sum=("High_Bid_Sum", "sum"),
        highbid_count=("High_Bid_Count", "sum"),

        floor_sum=("Floor_Sum", "sum"),
        floor_count=("Floor_Count", "sum"),
    )
    .reset_index()
)

del expanded
gc.collect()

agg["conversion"] = np.where(
    agg["registered"] > 0,
    agg["sold"] / agg["registered"],
    np.nan
)

agg["rpu"] = np.where(
    agg["sold"] > 0,
    agg["buyfee_revenue"] / agg["sold"],
    np.nan
)

agg["buyfee_coverage"] = np.where(
    agg["sold"] > 0,
    agg["sold_with_buyfee"] / agg["sold"],
    np.nan
)

def calc_avg(sum_col, count_col):
    return np.where(
        agg[count_col] > 0,
        agg[sum_col] / agg[count_col],
        np.nan
    )

agg["avg_saleamt"] = calc_avg("saleamt_sum", "saleamt_count")
agg["avg_age"] = calc_avg("age_sum", "age_count")
agg["avg_mileage"] = calc_avg("mileage_sum", "mileage_count")
agg["avg_condition"] = calc_avg("condition_sum", "condition_count")
agg["avg_high_bid"] = calc_avg("highbid_sum", "highbid_count")
agg["avg_floor"] = calc_avg("floor_sum", "floor_count")

# Keep only quality-qualified periods
agg = agg[
    (agg["observed_weeks"] >= MIN_OBSERVED_WEEKS)
    &
    (agg["registered"] >= MIN_REGISTERED_PRE)
].copy()

print("Qualified period rows:", f"{len(agg):,}")


Qualified period rows: 181,554


## 7. Pivot PRE and POST into one snapshot row

In [9]:

id_cols = [
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_DATE",
    "Event_Week",
    "VRLOCATION",
    "LANE",
    "price_tier"
]

value_cols = [
    "observed_weeks",
    "registered",
    "sold",
    "conversion",
    "rpu",
    "buyfee_coverage",
    "avg_saleamt",
    "avg_age",
    "avg_mileage",
    "avg_condition",
    "avg_high_bid",
    "avg_floor",
]

pre = agg[agg["Period"].eq("PRE")][id_cols + value_cols].copy()
post = agg[agg["Period"].eq("POST")][id_cols + value_cols].copy()

pre = pre.rename(
    columns={c: f"pre_{c}" for c in value_cols}
)

post = post.rename(
    columns={c: f"post_{c}" for c in value_cols}
)

snapshots = pre.merge(
    post,
    on=id_cols,
    how="inner"
)

snapshots = snapshots[
    (snapshots["pre_observed_weeks"] >= MIN_OBSERVED_WEEKS)
    &
    (snapshots["post_observed_weeks"] >= MIN_OBSERVED_WEEKS)
    &
    (snapshots["pre_registered"] >= MIN_REGISTERED_PRE)
    &
    (snapshots["post_registered"] >= MIN_REGISTERED_POST)
].copy()

snapshots.to_csv(
    OUTPUT_DIR / "02_EVENT_DATE_SEGMENT_SNAPSHOTS.csv",
    index=False
)

print("Snapshots:", f"{len(snapshots):,}")
print("Locations:", snapshots["VRLOCATION"].nunique())
print("Corporate dates:", snapshots["Corporate_Change_ID"].nunique())


Snapshots: 73,237
Locations: 52
Corporate dates: 37


## 8. Flag treated and control candidates

In [10]:

treated_key = strict[
    [
        "Corporate_Change_ID",
        "VRLOCATION",
        "LANE",
        "price_tier",
        "Segment_Change_ID",
        "BUYFEE_CHANGE_AMOUNT",
        "PREVIOUS_BUYFEE",
        "FEE_CHANGE_PCT"
    ]
].drop_duplicates()

snapshots = snapshots.merge(
    treated_key,
    on=[
        "Corporate_Change_ID",
        "VRLOCATION",
        "LANE",
        "price_tier"
    ],
    how="left"
)

snapshots["Is_Treated"] = snapshots["Segment_Change_ID"].notna()

# Contaminated controls: any segment with a fee change within +/-8 weeks
event_dates = (
    events[
        ["VRLOCATION", "LANE", "price_tier", "BUYFEE_CHANGE_DATE"]
    ]
    .dropna()
    .copy()
)

event_dates["BUYFEE_CHANGE_DATE"] = pd.to_datetime(
    event_dates["BUYFEE_CHANGE_DATE"],
    errors="coerce"
)

# Use merge to avoid row-wise apply across the full table.
candidate_controls = snapshots[
    ~snapshots["Is_Treated"]
].copy()

candidate_controls["_candidate_id"] = np.arange(len(candidate_controls))

potential = candidate_controls.merge(
    event_dates,
    on=["VRLOCATION", "LANE", "price_tier"],
    how="left",
    suffixes=("_event", "_other")
)

potential["Weeks_From_Other_Change"] = (
    (
        pd.to_datetime(potential["BUYFEE_CHANGE_DATE_other"])
        -
        pd.to_datetime(potential["BUYFEE_CHANGE_DATE_event"])
    )
    .dt.days
    .abs()
    / 7
)

contaminated_ids = set(
    potential.loc[
        potential["Weeks_From_Other_Change"] <= CONTROL_CHANGE_EXCLUSION_WEEKS,
        "_candidate_id"
    ]
)

candidate_controls["Eligible_Control"] = ~candidate_controls["_candidate_id"].isin(
    contaminated_ids
)

eligible_controls = candidate_controls[
    candidate_controls["Eligible_Control"]
].drop(columns=["_candidate_id"])

treated_snapshots = snapshots[
    snapshots["Is_Treated"]
].copy()

print("Treated snapshots:", len(treated_snapshots))
print("Eligible controls:", len(eligible_controls))


Treated snapshots: 1843
Eligible controls: 62057


## 9. Match controls

In [11]:

match_features = [
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
    "pre_avg_age",
    "pre_avg_mileage",
    "pre_avg_condition",
    "pre_avg_high_bid",
    "pre_avg_floor",
]

match_rows = []

for corp_id, treated_group in treated_snapshots.groupby("Corporate_Change_ID"):

    controls_date = eligible_controls[
        eligible_controls["Corporate_Change_ID"].eq(corp_id)
    ].copy()

    if controls_date.empty:
        continue

    for _, tr in treated_group.iterrows():

        candidates = controls_date

        if EXCLUDE_SAME_LOCATION:
            cross = candidates[
                candidates["VRLOCATION"].ne(tr["VRLOCATION"])
            ]
            if len(cross) >= N_CONTROLS:
                candidates = cross

        if pd.notna(tr["pre_avg_saleamt"]):
            near = candidates[
                candidates["pre_avg_saleamt"].between(
                    tr["pre_avg_saleamt"] - PRICE_TOLERANCE,
                    tr["pre_avg_saleamt"] + PRICE_TOLERANCE
                )
            ]
            if len(near) >= N_CONTROLS:
                candidates = near

        if candidates.empty:
            continue

        combo = pd.concat(
            [pd.DataFrame([tr]), candidates],
            ignore_index=True
        )

        X = combo[match_features].apply(
            pd.to_numeric,
            errors="coerce"
        )

        for c in match_features:
            med = X[c].median()
            X[c] = X[c].fillna(0 if pd.isna(med) else med)

        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)

        n_neighbors = min(N_CONTROLS, len(candidates))

        nn = NearestNeighbors(
            n_neighbors=n_neighbors,
            metric="euclidean"
        )

        nn.fit(Xs[1:])

        dist, idx = nn.kneighbors(Xs[[0]])

        selected = candidates.iloc[idx[0]].copy()
        selected["Match_Distance"] = dist[0]

        selected = selected[
            selected["Match_Distance"] <= MAX_MATCH_DISTANCE
        ]

        for rank, (_, ctrl) in enumerate(selected.iterrows(), start=1):

            row = {
                "Segment_Change_ID": tr["Segment_Change_ID"],
                "Corporate_Change_ID": corp_id,
                "BUYFEE_CHANGE_AMOUNT": tr["BUYFEE_CHANGE_AMOUNT"],
                "PREVIOUS_BUYFEE": tr["PREVIOUS_BUYFEE"],
                "FEE_CHANGE_PCT": tr["FEE_CHANGE_PCT"],

                "Treated_Location": tr["VRLOCATION"],
                "Treated_Lane": tr["LANE"],
                "Treated_Price_Tier": tr["price_tier"],

                "Control_Rank": rank,
                "Control_Location": ctrl["VRLOCATION"],
                "Control_Lane": ctrl["LANE"],
                "Control_Price_Tier": ctrl["price_tier"],
                "Match_Distance": ctrl["Match_Distance"],
            }

            for c in value_cols:
                row[f"T_pre_{c}"] = tr[f"pre_{c}"]
                row[f"T_post_{c}"] = tr[f"post_{c}"]
                row[f"C_pre_{c}"] = ctrl[f"pre_{c}"]
                row[f"C_post_{c}"] = ctrl[f"post_{c}"]

            match_rows.append(row)

matched_pairs = pd.DataFrame(match_rows)

matched_pairs.to_csv(
    OUTPUT_DIR / "03_MATCHED_CONTROL_PAIRS.csv",
    index=False
)

print("Matched pair rows:", len(matched_pairs))
print("Matched treated events:", matched_pairs["Segment_Change_ID"].nunique())
print(
    "Match rate:",
    matched_pairs["Segment_Change_ID"].nunique()
    / strict["Segment_Change_ID"].nunique()
)


Matched pair rows: 5435
Matched treated events: 1824
Match rate: 0.9896907216494846


## 10. Compute Difference-in-Differences

In [12]:

matched_pairs["T_Volume_Change_Pct"] = np.where(
    matched_pairs["T_pre_sold"] > 0,
    matched_pairs["T_post_sold"] / matched_pairs["T_pre_sold"] - 1,
    np.nan
)

matched_pairs["C_Volume_Change_Pct"] = np.where(
    matched_pairs["C_pre_sold"] > 0,
    matched_pairs["C_post_sold"] / matched_pairs["C_pre_sold"] - 1,
    np.nan
)

matched_pairs["DiD_Volume_Change_Pct"] = (
    matched_pairs["T_Volume_Change_Pct"]
    - matched_pairs["C_Volume_Change_Pct"]
)

matched_pairs["T_Conversion_Change_pp"] = (
    matched_pairs["T_post_conversion"]
    - matched_pairs["T_pre_conversion"]
) * 100

matched_pairs["C_Conversion_Change_pp"] = (
    matched_pairs["C_post_conversion"]
    - matched_pairs["C_pre_conversion"]
) * 100

matched_pairs["DiD_Conversion_Change_pp"] = (
    matched_pairs["T_Conversion_Change_pp"]
    - matched_pairs["C_Conversion_Change_pp"]
)

matched_pairs["T_RPU_Change_Dollar"] = (
    matched_pairs["T_post_rpu"]
    - matched_pairs["T_pre_rpu"]
)

matched_pairs["C_RPU_Change_Dollar"] = (
    matched_pairs["C_post_rpu"]
    - matched_pairs["C_pre_rpu"]
)

matched_pairs["DiD_RPU_Change_Dollar"] = (
    matched_pairs["T_RPU_Change_Dollar"]
    - matched_pairs["C_RPU_Change_Dollar"]
)

matched_pairs["DiD_Fee_Pass_Through"] = np.where(
    matched_pairs["BUYFEE_CHANGE_AMOUNT"] != 0,
    matched_pairs["DiD_RPU_Change_Dollar"]
    / matched_pairs["BUYFEE_CHANGE_AMOUNT"],
    np.nan
)

event_did = (
    matched_pairs.groupby(
        [
            "Segment_Change_ID",
            "Corporate_Change_ID",
            "BUYFEE_CHANGE_AMOUNT",
            "PREVIOUS_BUYFEE",
            "FEE_CHANGE_PCT",
            "Treated_Location",
            "Treated_Lane",
            "Treated_Price_Tier"
        ],
        as_index=False
    )
    .agg(
        Number_of_Controls=("Control_Rank", "count"),
        Median_Match_Distance=("Match_Distance", "median"),

        DiD_Volume_Change_Pct=("DiD_Volume_Change_Pct", "mean"),
        DiD_Conversion_Change_pp=("DiD_Conversion_Change_pp", "mean"),
        DiD_RPU_Change_Dollar=("DiD_RPU_Change_Dollar", "mean"),
        DiD_Fee_Pass_Through=("DiD_Fee_Pass_Through", "mean"),

        pre_registered=("T_pre_registered", "first"),
        pre_sold=("T_pre_sold", "first"),
        pre_conversion=("T_pre_conversion", "first"),
        pre_rpu=("T_pre_rpu", "first"),
        pre_avg_saleamt=("T_pre_avg_saleamt", "first"),
    )
)

event_did.to_csv(
    OUTPUT_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv",
    index=False
)

display(event_did.head())


,Segment_Change_ID,Corporate_Change_ID,BUYFEE_CHANGE_AMOUNT,PREVIOUS_BUYFEE,FEE_CHANGE_PCT,Treated_Location,Treated_Lane,Treated_Price_Tier,Number_of_Controls,Median_Match_Distance,DiD_Volume_Change_Pct,DiD_Conversion_Change_pp,DiD_RPU_Change_Dollar,DiD_Fee_Pass_Through,pre_registered,pre_sold,pre_conversion,pre_rpu,pre_avg_saleamt
0,ATLANTA_A_1000-2000_2024-12-01,2024-12-01,30.0,250.0,0.120000,ATLANTA,A,1000-2000,3,1.308194,0.034685,2.020544,25.014089,0.833803,57,38,0.666667,0.000000,1468.421053
1,ATLANTA_A_10000-11000_2024-12-01,2024-12-01,50.0,460.0,0.108696,ATLANTA,A,10000-11000,3,1.505764,-0.134921,-1.339671,30.993895,0.619878,32,14,0.437500,390.000000,10218.750000
2,ATLANTA_A_2000-3000_2024-12-01,2024-12-01,40.0,325.0,0.123077,ATLANTA,A,2000-3000,3,1.029995,-0.173794,-21.944969,1.047732,0.026193,55,34,0.618182,318.088235,2323.636364
3,ATLANTA_A_3000-4000_2024-12-01,2024-12-01,40.0,350.0,0.114286,ATLANTA,A,3000-4000,3,0.912546,0.081020,-4.635448,13.116380,0.327910,56,29,0.517857,343.103448,3425.000000
4,ATLANTA_A_4000-5000_2024-12-01,2024-12-01,40.0,365.0,0.109589,ATLANTA,A,4000-5000,3,1.079346,0.253197,1.935671,56.677123,1.416928,41,17,0.414634,302.941176,4336.585366


## 11. Scenario evidence

In [13]:

scenario_rows = []

for fee in [10, 15, 20, 25, 30]:

    d = event_did[
        event_did["BUYFEE_CHANGE_AMOUNT"].between(
            fee - 2.5,
            fee + 2.5
        )
    ]

    scenario_rows.append({
        "Fee_Increase": fee,
        "Matched_Events": len(d),
        "Unique_Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),

        "Median_DiD_Volume_Change_Pct": d["DiD_Volume_Change_Pct"].median(),
        "Median_DiD_Conversion_Change_pp": d["DiD_Conversion_Change_pp"].median(),
        "Median_DiD_RPU_Change_Dollar": d["DiD_RPU_Change_Dollar"].median(),
        "Median_DiD_Fee_Pass_Through": d["DiD_Fee_Pass_Through"].median(),
    })

scenario = pd.DataFrame(scenario_rows)

scenario.to_csv(
    OUTPUT_DIR / "05_DID_SCENARIO_EVIDENCE.csv",
    index=False
)

display(scenario)


,Fee_Increase,Matched_Events,Unique_Locations,Corporate_Change_Dates,Median_DiD_Volume_Change_Pct,Median_DiD_Conversion_Change_pp,Median_DiD_RPU_Change_Dollar,Median_DiD_Fee_Pass_Through
0,10,594,16,24,-0.003620,0.627338,7.630647,0.763065
1,15,84,14,10,0.044703,1.790793,10.481982,0.698799
2,20,133,12,10,-0.025353,1.658328,24.466728,1.223336
3,25,42,10,4,0.073548,0.901881,2.220601,0.088824
4,30,60,8,4,0.036691,-0.547817,2.512057,0.083735


## 12. Fixed-effects DiD summary

In [14]:

rows = []

target_map = {
    "Volume": "DiD_Volume_Change_Pct",
    "Conversion": "DiD_Conversion_Change_pp",
    "RPU": "DiD_RPU_Change_Dollar",
}

for name, target in target_map.items():

    d = event_did.dropna(
        subset=[
            target,
            "BUYFEE_CHANGE_AMOUNT",
            "Corporate_Change_ID",
            "Treated_Location"
        ]
    ).copy()

    if len(d) < 50:
        rows.append({
            "Target": name,
            "Rows": len(d),
            "Status": "INSUFFICIENT_ROWS"
        })
        continue

    formula = (
        f"{target} ~ BUYFEE_CHANGE_AMOUNT"
        " + FEE_CHANGE_PCT"
        " + pre_registered"
        " + pre_sold"
        " + pre_conversion"
        " + pre_avg_saleamt"
        " + C(Treated_Location)"
        " + C(Corporate_Change_ID)"
    )

    try:
        model = smf.ols(
            formula=formula,
            data=d
        ).fit(
            cov_type="cluster",
            cov_kwds={
                "groups": d["Corporate_Change_ID"]
            }
        )
    except Exception:
        model = smf.ols(
            formula=formula,
            data=d
        ).fit(cov_type="HC3")

    coef = model.params.get(
        "BUYFEE_CHANGE_AMOUNT",
        np.nan
    )

    pval = model.pvalues.get(
        "BUYFEE_CHANGE_AMOUNT",
        np.nan
    )

    rows.append({
        "Target": name,
        "Rows": len(d),
        "Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),
        "R2": model.rsquared,
        "Fee_Dollar_Coefficient": coef,
        "Fee_Dollar_PValue": pval,
        "Status": "OK"
    })

fe = pd.DataFrame(rows)

fe.to_csv(
    OUTPUT_DIR / "06_DID_FIXED_EFFECTS_SUMMARY.csv",
    index=False
)

display(fe)


C:\Users\ViditSheth\AppData\Local\Temp\ipykernel_16368\995118298.py:43: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit(
C:\Users\ViditSheth\AppData\Local\Temp\ipykernel_16368\995118298.py:43: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit(
C:\Users\ViditSheth\AppData\Local\Temp\ipykernel_16368\995118298.py:43: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  ).fit(


,Target,Rows,Locations,Corporate_Change_Dates,R2,Fee_Dollar_Coefficient,Fee_Dollar_PValue,Status
0,Volume,1824,35,37,0.083147,-0.001110,0.465816,OK
1,Conversion,1824,35,37,0.122368,-0.007233,0.869949,OK
2,RPU,1824,35,37,0.222478,0.178344,0.360222,OK


## 13. Final compact summary

In [15]:

summary = pd.DataFrame([
    {
        "Metric": "Strict_Treated_Events",
        "Value": strict["Segment_Change_ID"].nunique()
    },
    {
        "Metric": "Matched_Treated_Events",
        "Value": matched_pairs["Segment_Change_ID"].nunique()
    },
    {
        "Metric": "Match_Rate",
        "Value": (
            matched_pairs["Segment_Change_ID"].nunique()
            / strict["Segment_Change_ID"].nunique()
        )
    },
    {
        "Metric": "Matched_Locations",
        "Value": event_did["Treated_Location"].nunique()
    },
    {
        "Metric": "Matched_Corporate_Change_Dates",
        "Value": event_did["Corporate_Change_ID"].nunique()
    },
])

summary.to_csv(
    OUTPUT_DIR / "07_V8_2_SUMMARY.csv",
    index=False
)

display(summary)


,Metric,Value
0,Strict_Treated_Events,1843.000000
1,Matched_Treated_Events,1824.000000
2,Match_Rate,0.989691
3,Matched_Locations,35.000000
4,Matched_Corporate_Change_Dates,37.000000



## Files to share back

Please send:

1. `07_V8_2_SUMMARY.csv`
2. `05_DID_SCENARIO_EVIDENCE.csv`
3. `06_DID_FIXED_EFFECTS_SUMMARY.csv`
4. `04_DID_EVENT_LEVEL_RESULTS.csv`

This version intentionally removes the most memory-intensive diagnostics from earlier V8 versions. Once the core DiD pipeline runs successfully, we can add extra diagnostics back in smaller steps.
